# Generative AI Assignment 04: MAE, VAE, Diffusion and RAG

Before you start: in the Kaggle settings turn on **GPU T4** and **Internet**. The UHaT dataset is downloaded by a cell below, so no dataset has to be added.

Run the cells from top to bottom. Every file of the project is written by a cell, so the notebook needs nothing else.
Training cells take a long time: run them one at a time and wait for each one to finish.

## 0. Setup

In [1]:
!pip install -q diffusers rouge-score

In [2]:
import os
for folder in ['starter', 'starter/data/uhat', 'part_a_mae', 'part_b_vae', 'part_c_diffusion', 'part_d_rag',
               'results', 'weights', 'cache', 'tools', 'app']:
    os.makedirs(folder, exist_ok=True)
print(os.getcwd())

/kaggle/working


In [3]:
!wget -q -O data.zip 'https://zenodo.org/records/3670611/files/data.zip?download=1'
!unzip -q -o data.zip -d starter/data/uhat
!ls starter/data/uhat

data


## 1. The given starter files (not changed)

In [4]:
%%writefile starter/bioasq.py
"""Part D data: biomedical questions + PubMed abstracts (BioASQ, mini version)."""
from datasets import load_dataset

BIOASQ_REPO = "enelpol/rag-mini-bioasq"

def load_corpus():
    """Returns {passage_id (int): passage text (str)} for all 40k abstracts."""
    ds = load_dataset(BIOASQ_REPO, "text-corpus")
    split = list(ds.keys())[0]             # the corpus has a single split
    return {int(r["id"]): r["passage"] for r in ds[split] if r["passage"]}

def load_questions(split="test", corpus=None):
    """Returns a list of dicts: {qid, question, answer, relevant_ids}.
    relevant_ids = the ids of the abstracts that answer the question (the
    ground truth for measuring retrieval). If `corpus` is given, ids that are
    not in the corpus are dropped, and questions left with none are skipped."""
    ds = load_dataset(BIOASQ_REPO, "question-answer-passages", split=split)
    out = []
    for r in ds:
        rel = [int(i) for i in r["relevant_passage_ids"]]
        if corpus is not None:
            rel = [i for i in rel if i in corpus]
        if rel:
            out.append({"qid": int(r["id"]), "question": r["question"],
                        "answer": r["answer"], "relevant_ids": rel})
    return out


Writing starter/bioasq.py


In [5]:
%%writefile starter/eurosat.py
"""Part A data: EuroSAT RGB satellite tiles (64 x 64, 10 land-cover classes)."""
import numpy as np
import torch
from datasets import load_dataset

EUROSAT_REPO = "blanchon/EuroSAT_RGB"
CLASS_NAMES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
               "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]
PATCH = 8                      # 64 / 8 = 8 x 8 = 64 patches per image

def load_eurosat_split(split):
    """split in {'train', 'validation', 'test'}.
    Returns images (N, 3, 64, 64) float32 in [0, 1] and labels (N,) int64."""
    ds = load_dataset(EUROSAT_REPO, split=split)
    imgs = np.stack([np.asarray(im.convert("RGB"), dtype=np.uint8) for im in ds["image"]])
    images = torch.from_numpy(imgs).permute(0, 3, 1, 2).float() / 255.0
    labels = torch.tensor(ds["label"], dtype=torch.long)
    return images, labels

def few_label_indices(labels, per_class=16, seed=42):
    """Indices of a fixed labelled subset: `per_class` images of each class.
    Every group gets the same 160 images."""
    rng = np.random.default_rng(seed)
    idx = []
    for c in range(len(CLASS_NAMES)):
        members = np.flatnonzero(labels.numpy() == c)
        idx.extend(rng.choice(members, size=per_class, replace=False).tolist())
    return torch.tensor(sorted(idx))

def patchify(images, p=PATCH):
    """(N, C, H, W) -> (N, L, p*p*C) with L = (H/p)*(W/p), patches in row-major order."""
    N, C, H, W = images.shape
    x = images.reshape(N, C, H // p, p, W // p, p)
    x = x.permute(0, 2, 4, 3, 5, 1)                 # N, H/p, W/p, p, p, C
    return x.reshape(N, (H // p) * (W // p), p * p * C)

def unpatchify(patches, p=PATCH, C=3):
    """Inverse of patchify: (N, L, p*p*C) -> (N, C, H, W)."""
    N, L, _ = patches.shape
    h = w = int(L ** 0.5)
    x = patches.reshape(N, h, w, p, p, C).permute(0, 5, 1, 3, 2, 4)
    return x.reshape(N, C, h * p, w * p)


Writing starter/eurosat.py


In [6]:
%%writefile starter/uhat.py
"""Parts B and C data: UHaT handwritten Urdu characters (28 x 28, grayscale).
Unzip data.zip from Zenodo (or the Kaggle copy) into data/uhat/ first."""
from pathlib import Path
import numpy as np
import torch
from PIL import Image

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def _find_split_dir(root, kind, split):
    """Find the folder whose name mentions both `kind` ('char'/'digit') and
    `split` ('train'/'test') and that contains one sub-folder per class."""
    for d in sorted(Path(root).rglob("*")):
        name = d.name.lower()
        if d.is_dir() and kind in name and split in name:
            if any(c.is_dir() for c in d.iterdir()):
                return d
    found = [str(d) for d in Path(root).rglob("*") if d.is_dir()][:20]
    raise FileNotFoundError(f"No '{kind}'/'{split}' folder under {root}. Folders seen: {found}")

def load_uhat_characters(root="data/uhat", split="train", class_names=None):
    """Returns images (N, 1, 28, 28) float32 in [0, 1] with WHITE strokes on a
    BLACK background, labels (N,) int64, and the list of class names.
    Pass the training class_names when loading 'test' so labels match."""
    split_dir = _find_split_dir(root, "char", split)
    if class_names is None:
        class_names = sorted(c.name.lower() for c in split_dir.iterdir() if c.is_dir())
    index = {name: i for i, name in enumerate(class_names)}
    images, labels = [], []
    for cdir in sorted(split_dir.iterdir()):
        if not cdir.is_dir() or cdir.name.lower() not in index:
            continue
        for f in sorted(cdir.iterdir()):
            if f.suffix.lower() in IMG_EXT:
                im = Image.open(f).convert("L").resize((28, 28))
                images.append(np.asarray(im, dtype=np.float32) / 255.0)
                labels.append(index[cdir.name.lower()])
    x = torch.from_numpy(np.stack(images)).unsqueeze(1)
    # Make polarity consistent: if most pixels are bright, the background is
    # white, so invert to get white strokes on black.
    flip = x.mean(dim=(1, 2, 3)) > 0.5
    x[flip] = 1.0 - x[flip]
    return x, torch.tensor(labels, dtype=torch.long), class_names


Writing starter/uhat.py


In [7]:
%%writefile starter/viz.py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

def save_grid(images, path, nrow=8, titles=None):
    """images: (N, C, H, W) in [0, 1]. Saves an nrow-wide grid to `path`."""
    images = images.detach().cpu().clamp(0, 1)
    n = images.shape[0]
    rows = (n + nrow - 1) // nrow
    fig, axes = plt.subplots(rows, nrow, figsize=(1.4 * nrow, 1.4 * rows), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            img = images[i].permute(1, 2, 0).numpy()
            ax.imshow(img.squeeze(-1) if img.shape[-1] == 1 else img, cmap="gray", vmin=0, vmax=1)
            if titles is not None:
                ax.set_title(titles[i], fontsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


Writing starter/viz.py


In [8]:
%%writefile starter/check_starter.py
import os
import torch
from viz import save_grid
from eurosat import load_eurosat_split, few_label_indices, patchify, unpatchify, CLASS_NAMES
from uhat import load_uhat_characters
from bioasq import load_corpus, load_questions

os.makedirs("results", exist_ok=True)

# ---- Part A: EuroSAT ----
x_tr, y_tr = load_eurosat_split("train")
x_te, y_te = load_eurosat_split("test")
few = few_label_indices(y_tr)
p = patchify(x_tr[:8])
print("EuroSAT train", tuple(x_tr.shape), "test", tuple(x_te.shape))
print("few-label subset:", len(few), "images; patches per image:", tuple(p.shape))
assert torch.equal(unpatchify(p), x_tr[:8])
save_grid(x_tr[:32], "results/eurosat_examples.png", nrow=8,
          titles=[CLASS_NAMES[i] for i in y_tr[:32]])

# ---- Parts B and C: UHaT characters ----
c_tr, l_tr, names = load_uhat_characters(split="train")
c_te, l_te, _ = load_uhat_characters(split="test", class_names=names)
print("UHaT train", tuple(c_tr.shape), "test", tuple(c_te.shape), "classes:", len(names))
print("class names:", names)
pick = torch.randperm(len(c_tr))[:64]
save_grid(c_tr[pick], "results/uhat_examples.png", nrow=8, titles=[names[i] for i in l_tr[pick]])

# ---- Part D: BioASQ ----
corpus = load_corpus()
qs = load_questions("test", corpus)
print("BioASQ corpus:", len(corpus), "abstracts; test questions:", len(qs))
print("example:", qs[0]["question"], "| relevant ids:", qs[0]["relevant_ids"][:5])


Writing starter/check_starter.py


## 2. Shared helpers

In [9]:
%%writefile common.py
import os
import sys
import json
import time
import torch

sys.path.append("starter")

device = "cuda" if torch.cuda.is_available() else "cpu"
UHAT_ROOT = "starter/data/uhat"


def make_folders():
    os.makedirs("results", exist_ok=True)
    os.makedirs("weights", exist_ok=True)
    os.makedirs("cache", exist_ok=True)


def save_json(obj, path):
    file = open(path, "w")
    json.dump(obj, file, indent=1)
    file.close()


def load_json(path):
    file = open(path)
    obj = json.load(file)
    file.close()
    return obj


def mean(values):
    return sum(values) / len(values)


def count_params(model):
    total = 0
    for p in model.parameters():
        total = total + p.numel()
    return total


def gpu_name():
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return "cpu"


def batches(n, batch_size, shuffle=True):
    if shuffle:
        order = torch.randperm(n)
    else:
        order = torch.arange(n)
    for start in range(0, n, batch_size):
        yield order[start:start + batch_size]


def load_uhat():
    saved_file = "cache/uhat.pt"
    if os.path.exists(saved_file):
        return torch.load(saved_file)
    from uhat import load_uhat_characters
    x_train, y_train, names = load_uhat_characters(root=UHAT_ROOT, split="train")
    x_test, y_test, _ = load_uhat_characters(root=UHAT_ROOT, split="test", class_names=names)
    os.makedirs("cache", exist_ok=True)
    torch.save((x_train, y_train, x_test, y_test, names), saved_file)
    return x_train, y_train, x_test, y_test, names


Writing common.py


## 3. Task 0: run the starter code

In [10]:
%cd starter
!python check_starter.py
%cd ..

/kaggle/working/starter
README.md: 100%|███████████████████████████| 3.38k/3.38k [00:00<00:00, 7.03MB/s]

data/train-00000-of-00001.parquet: downloading bytes: █████▏| 90.2MB, 6.79MB/s  
data/train-00000-of-00001.parquet: downloading bytes: ██████|  104MB, 9.89MB/s  
data/train-00000-of-00001.parquet: reconstructing file: 100%|█|  105MB /  105MB,

data/test-00000-of-00001.parquet: downloading bytes: ██████▊| 33.6MB, 1.70MB/s  
data/test-00000-of-00001.parquet: downloading bytes: ███████| 34.7MB, 3.35MB/s  
data/test-00000-of-00001.parquet: reconstructing file: 100%|█| 34.8MB / 34.8MB, 

data/validation-00000-of-00001.parquet: downloading bytes: ▋| 25.7MB, 1.03MB/s  
data/validation-00000-of-00001.parquet: downloading bytes: █| 34.8MB, 3.37MB/s  
data/validation-00000-of-00001.parquet: reconstructing file: 100%|█| 34.8MB / 34
Generating test split: 100%|██████| 5400/5400 [00:00<00:00, 88412.13 examples/s]
Generating validation split: 100%|█| 5400/5400 [00:00<00:00, 90039.08 examples/s
E

## 4. Part A: masked autoencoder

In [11]:
%%writefile part_a_mae/mae_model.py
import torch
import torch.nn as nn


class MAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_patches = 64
        self.num_keep = 16

        self.patch_embed = nn.Linear(192, 192)
        self.pos_embed = nn.Parameter(torch.randn(1, 64, 192) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=192, nhead=3, dim_feedforward=768, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=6, enable_nested_tensor=False)
        self.enc_norm = nn.LayerNorm(192)

        self.dec_embed = nn.Linear(192, 128)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, 128))
        self.dec_pos_embed = nn.Parameter(torch.randn(1, 64, 128) * 0.02)
        dec_layer = nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=512, batch_first=True)
        self.decoder = nn.TransformerEncoder(dec_layer, num_layers=2, enable_nested_tensor=False)
        self.dec_out = nn.Linear(128, 192)

    def random_masking(self, batch_size, device):
        noise = torch.rand(batch_size, self.num_patches, device=device)
        order = noise.argsort(dim=1)
        keep_idx = order[:, :self.num_keep]
        mask = torch.ones(batch_size, self.num_patches, device=device)
        mask.scatter_(1, keep_idx, 0.0)
        return keep_idx, mask

    def forward(self, patches):
        batch_size = patches.shape[0]
        keep_idx, mask = self.random_masking(batch_size, patches.device)

        x = self.patch_embed(patches) + self.pos_embed
        index = keep_idx.unsqueeze(-1).expand(-1, -1, 192)
        visible = torch.gather(x, 1, index)
        encoded = self.enc_norm(self.encoder(visible))

        d = self.dec_embed(encoded)
        full = self.mask_token.expand(batch_size, self.num_patches, 128)
        index = keep_idx.unsqueeze(-1).expand(-1, -1, 128)
        full = full.scatter(1, index, d)
        full = full + self.dec_pos_embed
        full = self.decoder(full)
        pred = self.dec_out(full)

        loss = self.masked_loss(pred, patches, mask)
        return loss, pred, mask

    def masked_loss(self, pred, patches, mask):
        per_patch = ((pred - patches) ** 2).mean(dim=-1)
        return (per_patch * mask).sum() / mask.sum()

    def features(self, patches):
        x = self.patch_embed(patches) + self.pos_embed
        encoded = self.enc_norm(self.encoder(x))
        return encoded.mean(dim=1)


Writing part_a_mae/mae_model.py


In [12]:
%%writefile part_a_mae/pretrain.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_a_mae")

import time
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import *
from eurosat import load_eurosat_split, patchify
from mae_model import MAE

torch.manual_seed(0)
make_folders()

EPOCHS = 100
BATCH_SIZE = 256

images, labels = load_eurosat_split("train")
patches = patchify(images)
print("training images:", patches.shape[0], "| patches per image:", patches.shape[1])

model = MAE().to(device)
print("parameters:", count_params(model))

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)

epoch_losses = []
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    losses = []
    for idx in batches(patches.shape[0], BATCH_SIZE):
        batch = patches[idx].to(device)
        loss, pred, mask = model(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    epoch_loss = mean(losses)
    epoch_losses.append(epoch_loss)
    print("epoch", epoch + 1, "| loss", round(epoch_loss, 5))

seconds = time.time() - start_time
print("time in minutes:", round(seconds / 60, 1))

torch.save(model.state_dict(), "weights/mae.pt")

save_json({"epoch_losses": epoch_losses, "params": count_params(model), "seconds": seconds,
           "gpu": gpu_name(), "train_images": patches.shape[0]}, "results/mae_pretrain.json")

plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), epoch_losses)
plt.xlabel("epoch")
plt.ylabel("masked MSE loss")
plt.title("MAE pretraining loss")
plt.tight_layout()
plt.savefig("results/fig_mae_loss.png", dpi=150)
print("saved results/fig_mae_loss.png")


Writing part_a_mae/pretrain.py


In [13]:
!python part_a_mae/pretrain.py

training images: 16200 | patches per image: 64
parameters: 3173248
epoch 1 | loss 0.05303
epoch 2 | loss 0.02328
epoch 3 | loss 0.01507
epoch 4 | loss 0.0128
epoch 5 | loss 0.01113
epoch 6 | loss 0.01051
epoch 7 | loss 0.00972
epoch 8 | loss 0.00945
epoch 9 | loss 0.00929
epoch 10 | loss 0.00896
epoch 11 | loss 0.00893
epoch 12 | loss 0.00881
epoch 13 | loss 0.00874
epoch 14 | loss 0.00855
epoch 15 | loss 0.00854
epoch 16 | loss 0.00853
epoch 17 | loss 0.00848
epoch 18 | loss 0.00841
epoch 19 | loss 0.00839
epoch 20 | loss 0.00836
epoch 21 | loss 0.00836
epoch 22 | loss 0.00804
epoch 23 | loss 0.00795
epoch 24 | loss 0.00791
epoch 25 | loss 0.00784
epoch 26 | loss 0.00782
epoch 27 | loss 0.00781
epoch 28 | loss 0.00787
epoch 29 | loss 0.0078
epoch 30 | loss 0.00781
epoch 31 | loss 0.00772
epoch 32 | loss 0.00768
epoch 33 | loss 0.00765
epoch 34 | loss 0.00769
epoch 35 | loss 0.00763
epoch 36 | loss 0.00762
epoch 37 | loss 0.00765
epoch 38 | loss 0.00761
epoch 39 | loss 0.0076
epoch 40 

In [14]:
%%writefile part_a_mae/reconstruct.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_a_mae")

import torch
from common import *
from viz import save_grid
from eurosat import load_eurosat_split, patchify, unpatchify
from mae_model import MAE

torch.manual_seed(1)
make_folders()

images, labels = load_eurosat_split("test")
model = MAE().to(device)
model.load_state_dict(torch.load("weights/mae.pt", map_location=device))
model.eval()

original = images[:8]
patches = patchify(original)

with torch.no_grad():
    loss, pred, mask = model(patches.to(device))
pred = pred.cpu()
mask = mask.cpu()

mask3 = mask.unsqueeze(-1)
masked_input = patches * (1 - mask3) + 0.5 * mask3
reconstruction = patches * (1 - mask3) + pred * mask3

row1 = unpatchify(masked_input)
row2 = unpatchify(reconstruction)
row3 = original

grid = torch.cat([row1, row2, row3], dim=0)
save_grid(grid, "results/fig_mae_reconstructions.png", nrow=8)
print("saved results/fig_mae_reconstructions.png (rows: masked input, reconstruction, original)")


Writing part_a_mae/reconstruct.py


In [15]:
!python part_a_mae/reconstruct.py

saved results/fig_mae_reconstructions.png (rows: masked input, reconstruction, original)


In [16]:
%%writefile part_a_mae/classify.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_a_mae")

import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import *
from eurosat import load_eurosat_split, few_label_indices, patchify, CLASS_NAMES
from mae_model import MAE

torch.manual_seed(2)
make_folders()

x_train, y_train = load_eurosat_split("train")
x_test, y_test = load_eurosat_split("test")
few = few_label_indices(y_train)
print("labelled images used:", len(few))

model = MAE().to(device)
model.load_state_dict(torch.load("weights/mae.pt", map_location=device))
model.eval()
for p in model.parameters():
    p.requires_grad = False


def get_features(images):
    patches = patchify(images)
    all_features = []
    with torch.no_grad():
        for idx in batches(patches.shape[0], 500, shuffle=False):
            all_features.append(model.features(patches[idx].to(device)))
    return torch.cat(all_features)


train_features = get_features(x_train[few])
train_labels = y_train[few].to(device)
test_features = get_features(x_test)
test_labels = y_test.to(device)

classifier = nn.Linear(192, 10).to(device)
optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    logits = classifier(train_features)
    loss = loss_fn(logits, train_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print("epoch", epoch + 1, "| loss", round(loss.item(), 4))

with torch.no_grad():
    predictions = classifier(test_features).argmax(dim=1)

accuracy = (predictions == test_labels).float().mean().item()
print("test accuracy:", round(accuracy, 4), "(chance is 0.1)")

confusion = torch.zeros(10, 10, dtype=torch.long)
for true, pred in zip(test_labels.tolist(), predictions.tolist()):
    confusion[true, pred] += 1

pairs = []
for i in range(10):
    for j in range(10):
        if i != j:
            pairs.append((confusion[i, j].item(), CLASS_NAMES[i], CLASS_NAMES[j]))
pairs.sort(reverse=True)
print("most confused (count, true class, predicted class):")
for pair in pairs[:5]:
    print(pair)

torch.save(classifier.state_dict(), "weights/mae_classifier.pt")
save_json({"test_accuracy": accuracy, "confusion": confusion.tolist(), "class_names": CLASS_NAMES,
           "test_images": len(y_test), "labelled_images": len(few),
           "most_confused": pairs[:5]}, "results/mae_classifier.json")

plt.figure(figsize=(7, 6))
plt.imshow(confusion.numpy(), cmap="Blues")
plt.xticks(range(10), CLASS_NAMES, rotation=90)
plt.yticks(range(10), CLASS_NAMES)
for i in range(10):
    for j in range(10):
        plt.text(j, i, str(confusion[i, j].item()), ha="center", va="center", fontsize=7)
plt.xlabel("predicted")
plt.ylabel("true")
plt.title("Confusion matrix (accuracy " + str(round(accuracy, 3)) + ")")
plt.tight_layout()
plt.savefig("results/fig_mae_confusion.png", dpi=150)
print("saved results/fig_mae_confusion.png")


Writing part_a_mae/classify.py


In [17]:
!python part_a_mae/classify.py

labelled images used: 160
epoch 50 | loss 1.3705
epoch 100 | loss 1.1456
epoch 150 | loss 1.028
epoch 200 | loss 0.9458
test accuracy: 0.5281 (chance is 0.1)
most confused (count, true class, predicted class):
(153, 'SeaLake', 'Pasture')
(127, 'PermanentCrop', 'HerbaceousVegetation')
(114, 'AnnualCrop', 'PermanentCrop')
(107, 'Highway', 'River')
(97, 'River', 'Highway')
saved results/fig_mae_confusion.png


## 5. Part B: variational autoencoder

In [18]:
%%writefile part_b_vae/vae_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class VAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.latent_dim = latent_dim

        self.enc_conv1 = nn.Conv2d(1, 32, 4, stride=2, padding=1)
        self.enc_conv2 = nn.Conv2d(32, 64, 4, stride=2, padding=1)
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)

        self.dec_fc = nn.Linear(latent_dim, 64 * 7 * 7)
        self.dec_conv1 = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.dec_conv2 = nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1)

    def encode(self, x):
        h = F.relu(self.enc_conv1(x))
        h = F.relu(self.enc_conv2(h))
        h = h.flatten(start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = self.dec_fc(z)
        h = h.reshape(-1, 64, 7, 7)
        h = F.relu(self.dec_conv1(h))
        return self.dec_conv2(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


def kl_term(mu, logvar):
    kl = -0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(dim=1)
    return kl.mean()


def recon_term(logits, x):
    bce = F.binary_cross_entropy_with_logits(logits, x, reduction="none")
    return bce.sum(dim=(1, 2, 3)).mean()


def vae_loss(logits, x, mu, logvar):
    recon = recon_term(logits, x)
    kl = kl_term(mu, logvar)
    return recon + kl, recon, kl


Writing part_b_vae/vae_model.py


In [19]:
%%writefile part_b_vae/train.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_b_vae")

import time
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import *
from vae_model import VAE, vae_loss

torch.manual_seed(3)
make_folders()

EPOCHS = 30
BATCH_SIZE = 128

x_train, y_train, x_test, y_test, names = load_uhat()
print("UHaT train:", tuple(x_train.shape), "| test:", tuple(x_test.shape), "| classes:", len(names))

model = VAE().to(device)
print("parameters:", count_params(model))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def test_losses():
    model.eval()
    total = 0.0
    recon_sum = 0.0
    kl_sum = 0.0
    with torch.no_grad():
        for idx in batches(x_test.shape[0], 500, shuffle=False):
            x = x_test[idx].to(device)
            logits, mu, logvar = model(x)
            loss, recon, kl = vae_loss(logits, x, mu, logvar)
            n = x.shape[0]
            total += loss.item() * n
            recon_sum += recon.item() * n
            kl_sum += kl.item() * n
    n_all = x_test.shape[0]
    return recon_sum / n_all, kl_sum / n_all, total / n_all


log = {"train_recon": [], "train_kl": [], "train_total": [],
       "test_recon": [], "test_kl": [], "test_total": []}
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    recon_list = []
    kl_list = []
    total_list = []
    for idx in batches(x_train.shape[0], BATCH_SIZE):
        x = x_train[idx].to(device)
        logits, mu, logvar = model(x)
        loss, recon, kl = vae_loss(logits, x, mu, logvar)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        recon_list.append(recon.item())
        kl_list.append(kl.item())
        total_list.append(loss.item())

    test_recon, test_kl, test_total = test_losses()
    log["train_recon"].append(mean(recon_list))
    log["train_kl"].append(mean(kl_list))
    log["train_total"].append(mean(total_list))
    log["test_recon"].append(test_recon)
    log["test_kl"].append(test_kl)
    log["test_total"].append(test_total)
    print("epoch", epoch + 1, "| train recon", round(mean(recon_list), 2), "kl", round(mean(kl_list), 2),
          "| test recon", round(test_recon, 2), "kl", round(test_kl, 2), "total", round(test_total, 2))

seconds = time.time() - start_time
print("time in minutes:", round(seconds / 60, 1))

torch.save(model.state_dict(), "weights/vae.pt")

log["params"] = count_params(model)
log["seconds"] = seconds
log["gpu"] = gpu_name()
log["train_images"] = x_train.shape[0]
log["test_images"] = x_test.shape[0]
log["classes"] = len(names)
log["class_names"] = names
save_json(log, "results/vae_train.json")

epochs = range(1, EPOCHS + 1)
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(epochs, log["train_recon"], label="train")
plt.plot(epochs, log["test_recon"], label="test")
plt.title("reconstruction term")
plt.xlabel("epoch")
plt.legend()
plt.subplot(1, 3, 2)
plt.plot(epochs, log["train_kl"], label="train")
plt.plot(epochs, log["test_kl"], label="test")
plt.title("KL term")
plt.xlabel("epoch")
plt.legend()
plt.subplot(1, 3, 3)
plt.plot(epochs, log["train_total"], label="train")
plt.plot(epochs, log["test_total"], label="test")
plt.title("negative ELBO")
plt.xlabel("epoch")
plt.legend()
plt.tight_layout()
plt.savefig("results/fig_vae_loss.png", dpi=150)
print("saved results/fig_vae_loss.png")


Writing part_b_vae/train.py


In [20]:
!python part_b_vae/train.py

UHaT train: (28328, 1, 28, 28) | test: (4747, 1, 28, 28) | classes: 40
parameters: 220385
epoch 1 | train recon 190.03 kl 14.27 | test recon 146.5 kl 17.46 total 163.96
epoch 2 | train recon 140.04 kl 18.54 | test recon 134.42 kl 18.33 total 152.75
epoch 3 | train recon 131.73 kl 19.69 | test recon 128.12 kl 20.06 total 148.17
epoch 4 | train recon 127.24 kl 20.34 | test recon 124.18 kl 20.62 total 144.8
epoch 5 | train recon 124.05 kl 20.92 | test recon 122.28 kl 20.9 total 143.18
epoch 6 | train recon 121.68 kl 21.35 | test recon 119.83 kl 21.57 total 141.41
epoch 7 | train recon 119.93 kl 21.73 | test recon 117.98 kl 22.2 total 140.18
epoch 8 | train recon 118.54 kl 22.07 | test recon 117.75 kl 21.56 total 139.31
epoch 9 | train recon 117.29 kl 22.36 | test recon 116.26 kl 22.38 total 138.64
epoch 10 | train recon 116.24 kl 22.6 | test recon 115.29 kl 22.62 total 137.9
epoch 11 | train recon 115.24 kl 22.91 | test recon 113.8 kl 23.23 total 137.02
epoch 12 | train recon 114.2 kl 23.

In [21]:
%%writefile part_b_vae/visualize.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_b_vae")

import torch
from common import *
from viz import save_grid
from vae_model import VAE

torch.manual_seed(4)
make_folders()

x_train, y_train, x_test, y_test, names = load_uhat()

model = VAE().to(device)
model.load_state_dict(torch.load("weights/vae.pt", map_location=device))
model.eval()

pick = torch.randperm(x_test.shape[0])[:16]
x = x_test[pick].to(device)
with torch.no_grad():
    mu, logvar = model.encode(x)
    recon = torch.sigmoid(model.decode(mu))
grid = torch.cat([x.cpu(), recon.cpu()], dim=0)
save_grid(grid, "results/fig_vae_reconstructions.png", nrow=16)
print("saved results/fig_vae_reconstructions.png (top row: test images, bottom row: reconstructions)")

z = torch.randn(64, 16).to(device)
with torch.no_grad():
    samples = torch.sigmoid(model.decode(z))
save_grid(samples.cpu(), "results/fig_vae_samples.png", nrow=8)
print("saved results/fig_vae_samples.png")

first = 0
second = 1
while y_test[pick[second]] == y_test[pick[first]]:
    second = second + 1
a = x_test[pick[first]].unsqueeze(0).to(device)
b = x_test[pick[second]].unsqueeze(0).to(device)
with torch.no_grad():
    mu_a, _ = model.encode(a)
    mu_b, _ = model.encode(b)
    steps = []
    for i in range(10):
        t = i / 9
        z = (1 - t) * mu_a + t * mu_b
        steps.append(torch.sigmoid(model.decode(z)))
steps = torch.cat(steps, dim=0)
save_grid(steps.cpu(), "results/fig_vae_interpolation.png", nrow=10)
print("interpolation between", names[y_test[pick[first]]], "and", names[y_test[pick[second]]])
print("saved results/fig_vae_interpolation.png")


Writing part_b_vae/visualize.py


In [22]:
!python part_b_vae/visualize.py

saved results/fig_vae_reconstructions.png (top row: test images, bottom row: reconstructions)
saved results/fig_vae_samples.png
interpolation between paa and meem
saved results/fig_vae_interpolation.png


## 6. Part C: diffusion

In [23]:
%%writefile part_c_diffusion/schedule.py
import torch

T = 1000

betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def add_noise(x0, t, noise):
    abar = alpha_bars[t.cpu()].to(x0.device).reshape(-1, 1, 1, 1)
    return abar.sqrt() * x0 + (1 - abar).sqrt() * noise


Writing part_c_diffusion/schedule.py


In [24]:
%%writefile part_c_diffusion/unet.py
from diffusers import UNet2DModel


def make_unet():
    unet = UNet2DModel(
        sample_size=28,
        in_channels=1,
        out_channels=1,
        layers_per_block=2,
        block_out_channels=(32, 64, 128),
        down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
        num_class_embeds=40,
    )
    return unet


Writing part_c_diffusion/unet.py


In [25]:
%%writefile part_c_diffusion/train.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_c_diffusion")

import time
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import *
from schedule import T, add_noise
from unet import make_unet

torch.manual_seed(5)
make_folders()

EPOCHS = 40
BATCH_SIZE = 128

x_train, y_train, x_test, y_test, names = load_uhat()
x_train = x_train * 2 - 1
print("UHaT train:", tuple(x_train.shape), "| classes:", len(names))

unet = make_unet().to(device)
print("parameters:", count_params(unet))
optimizer = torch.optim.Adam(unet.parameters(), lr=2e-4)

epoch_losses = []
start_time = time.time()

for epoch in range(EPOCHS):
    unet.train()
    losses = []
    for idx in batches(x_train.shape[0], BATCH_SIZE):
        x0 = x_train[idx].to(device)
        y = y_train[idx].to(device)
        t = torch.randint(0, T, (x0.shape[0],), device=device)
        noise = torch.randn_like(x0)
        x_t = add_noise(x0, t, noise)
        predicted = unet(x_t, t, class_labels=y).sample
        loss = F.mse_loss(predicted, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    epoch_loss = mean(losses)
    epoch_losses.append(epoch_loss)
    print("epoch", epoch + 1, "| loss", round(epoch_loss, 5))
    torch.save(unet.state_dict(), "weights/unet.pt")

seconds = time.time() - start_time
print("time in minutes:", round(seconds / 60, 1))

save_json({"epoch_losses": epoch_losses, "params": count_params(unet), "seconds": seconds,
           "gpu": gpu_name(), "train_images": x_train.shape[0], "test_images": x_test.shape[0],
           "classes": len(names), "class_names": names}, "results/diffusion_train.json")

plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), epoch_losses)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Diffusion training loss")
plt.tight_layout()
plt.savefig("results/fig_diffusion_loss.png", dpi=150)
print("saved results/fig_diffusion_loss.png")


Writing part_c_diffusion/train.py


In [26]:
!python part_c_diffusion/train.py

UHaT train: (28328, 1, 28, 28) | classes: 40
parameters: 4028353
epoch 1 | loss 0.12713
epoch 2 | loss 0.05949
epoch 3 | loss 0.04982
epoch 4 | loss 0.04282
epoch 5 | loss 0.03752
epoch 6 | loss 0.03389
epoch 7 | loss 0.03287
epoch 8 | loss 0.03118
epoch 9 | loss 0.02973
epoch 10 | loss 0.02845
epoch 11 | loss 0.02829
epoch 12 | loss 0.02789
epoch 13 | loss 0.02786
epoch 14 | loss 0.02707
epoch 15 | loss 0.02688
epoch 16 | loss 0.02708
epoch 17 | loss 0.02674
epoch 18 | loss 0.02549
epoch 19 | loss 0.02525
epoch 20 | loss 0.02518
epoch 21 | loss 0.02537
epoch 22 | loss 0.02466
epoch 23 | loss 0.02423
epoch 24 | loss 0.02444
epoch 25 | loss 0.02447
epoch 26 | loss 0.02379
epoch 27 | loss 0.02392
epoch 28 | loss 0.02439
epoch 29 | loss 0.02401
epoch 30 | loss 0.02391
epoch 31 | loss 0.02448
epoch 32 | loss 0.024
epoch 33 | loss 0.02367
epoch 34 | loss 0.02386
epoch 35 | loss 0.02333
epoch 36 | loss 0.02369
epoch 37 | loss 0.02304
epoch 38 | loss 0.02356
epoch 39 | loss 0.02348
epoch 40 |

In [27]:
%%writefile part_c_diffusion/sample.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_c_diffusion")

import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import *
from viz import save_grid
from schedule import T, betas, alphas, alpha_bars, add_noise
from unet import make_unet


def sample_images(unet, labels, keep_steps=None):
    n = labels.shape[0]
    x = torch.randn(n, 1, 28, 28, device=device)
    saved = {}
    with torch.no_grad():
        for t in reversed(range(T)):
            t_batch = torch.full((n,), t, device=device, dtype=torch.long)
            predicted = unet(x, t_batch, class_labels=labels).sample
            alpha = alphas[t].item()
            abar = alpha_bars[t].item()
            beta = betas[t].item()
            x = (1 / alpha ** 0.5) * (x - (1 - alpha) / (1 - abar) ** 0.5 * predicted)
            if t > 0:
                x = x + (beta ** 0.5) * torch.randn_like(x)
            if keep_steps is not None and t in keep_steps:
                saved[t] = x.clamp(-1, 1).cpu()
    return x.clamp(-1, 1), saved


def to_01(x):
    return (x + 1) / 2


if __name__ == "__main__":
    torch.manual_seed(6)
    make_folders()

    x_train, y_train, x_test, y_test, names = load_uhat()

    unet = make_unet().to(device)
    unet.load_state_dict(torch.load("weights/unet.pt", map_location=device))
    unet.eval()

    image = x_train[0:1] * 2 - 1
    times = [0, 100, 250, 500, 750, 999]
    noised = []
    noise = torch.randn_like(image)
    for t in times:
        noised.append(add_noise(image, torch.tensor([t]), noise))
    noised = torch.cat(noised, dim=0).clamp(-1, 1)
    save_grid(to_01(noised), "results/fig_diffusion_forward.png", nrow=6,
              titles=["t=" + str(t) for t in times])
    print("saved results/fig_diffusion_forward.png")

    labels = torch.arange(40).repeat_interleave(8).to(device)
    all_samples = []
    for start in range(0, 320, 160):
        samples, saved = sample_images(unet, labels[start:start + 160])
        all_samples.append(samples.cpu())
        print("sampled", start + 160, "of 320")
    all_samples = to_01(torch.cat(all_samples, dim=0))

    fig, axes = plt.subplots(40, 8, figsize=(8, 40))
    for row in range(40):
        for col in range(8):
            ax = axes[row, col]
            ax.imshow(all_samples[row * 8 + col, 0], cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([])
            ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(names[row], rotation=0, ha="right", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig("results/fig_diffusion_samples.png", dpi=100)
    plt.close()
    print("saved results/fig_diffusion_samples.png")

    keep = [999, 750, 500, 250, 100, 0]
    one_label = torch.tensor([0], device=device)
    final, saved = sample_images(unet, one_label, keep_steps=keep)
    steps = torch.cat([saved[t] for t in keep], dim=0)
    save_grid(to_01(steps), "results/fig_diffusion_denoising.png", nrow=6,
              titles=["t=" + str(t) for t in keep])
    print("saved results/fig_diffusion_denoising.png (character: " + names[0] + ")")


Writing part_c_diffusion/sample.py


In [28]:
!python part_c_diffusion/sample.py

saved results/fig_diffusion_forward.png
sampled 160 of 320
sampled 320 of 320
saved results/fig_diffusion_samples.png
saved results/fig_diffusion_denoising.png (character: alif)


In [29]:
%%writefile part_c_diffusion/blind_test.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_c_diffusion")

import csv
import random
import torch
from common import *
from viz import save_grid
from unet import make_unet
from sample import sample_images, to_01

torch.manual_seed(7)
random.seed(7)
make_folders()

x_train, y_train, x_test, y_test, names = load_uhat()

unet = make_unet().to(device)
unet.load_state_dict(torch.load("weights/unet.pt", map_location=device))
unet.eval()

labels = torch.arange(40).to(device)
samples, saved = sample_images(unet, labels)
samples = to_01(samples).cpu()

order = list(range(40))
random.shuffle(order)

shuffled = samples[order]
save_grid(shuffled, "results/blind_test_images.png", nrow=8, titles=[str(i + 1) for i in range(40)])
print("saved results/blind_test_images.png (images numbered 1 to 40)")

key = {}
for position in range(40):
    key[str(position + 1)] = names[order[position]]
save_json(key, "results/blind_test_key.json")

columns = ["kind", "item", "text", "member1_answer", "member2_answer",
           "member1_supported", "member2_supported", "member1_correct", "member2_correct"]

file = open("results/human_eval.csv", "w", newline="", encoding="utf-8")
writer = csv.DictWriter(file, fieldnames=columns)
writer.writeheader()
for position in range(40):
    row = {}
    for c in columns:
        row[c] = ""
    row["kind"] = "diffusion"
    row["item"] = position + 1
    writer.writerow(row)
file.close()

print("saved results/human_eval.csv")
print("each member writes the character name for every image in member1_answer / member2_answer")
print("class names:", names)
print("do not open results/blind_test_key.json until both members have finished")


Writing part_c_diffusion/blind_test.py


In [30]:
!python part_c_diffusion/blind_test.py

saved results/blind_test_images.png (images numbered 1 to 40)
saved results/human_eval.csv
each member writes the character name for every image in member1_answer / member2_answer
class names: ['alif', 'alif mad aa', 'ayn', 'baa', 'bari yaa', 'cheey', 'choti yaa', 'daal', 'dhaal', 'faa', 'gaaf', 'ghain', 'haa1', 'haa2', 'haa3', 'hamza', 'jeem', 'kaaf', 'khaa', 'laam', 'meem', 'noon', 'noonghunna', 'paa', 'qaaf', 'raa', 'rhraa', 'seen', 'seey', 'sheen', 'swaad', 'taa', 'ttaa', 'twa', 'waw', 'zaaa', 'zaal', 'zhaa', 'zwaa', 'zwaad']
do not open results/blind_test_key.json until both members have finished


## 7. Part D: retrieval augmented generation

In [31]:
%%writefile part_d_rag/rag.py
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

RETRIEVER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
CHUNK_WORDS = 150
OVERLAP_WORDS = 30


def split_into_chunks(text, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    words = text.split()
    step = size - overlap
    chunks = []
    start = 0
    while True:
        piece = words[start:start + size]
        chunks.append(" ".join(piece))
        if start + size >= len(words):
            break
        start = start + step
    return chunks


def load_retriever():
    tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_NAME)
    model = AutoModel.from_pretrained(RETRIEVER_NAME)
    model = model.to(device)
    model.eval()
    return tokenizer, model


def embed_texts(texts, tokenizer, model, batch_size=128):
    all_vectors = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
            encoded = encoded.to(device)
            hidden = model(**encoded).last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1).float()
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            pooled = F.normalize(pooled, p=2, dim=1)
            all_vectors.append(pooled.cpu())
    return torch.cat(all_vectors, dim=0)


class Index:
    def __init__(self, folder="cache"):
        self.embeddings = torch.load(folder + "/embeddings.pt").to(device)
        file = open(folder + "/chunks.json", encoding="utf-8")
        data = json.load(file)
        file.close()
        self.chunk_abstract_ids = data["abstract_ids"]
        self.chunk_texts = data["texts"]

        self.abstract_ids = sorted(set(self.chunk_abstract_ids))
        position = {}
        for i, abstract_id in enumerate(self.abstract_ids):
            position[abstract_id] = i
        mapped = []
        for abstract_id in self.chunk_abstract_ids:
            mapped.append(position[abstract_id])
        self.chunk_to_abstract = torch.tensor(mapped, device=device)

    def top_abstracts(self, query_vector, k):
        scores = self.embeddings @ query_vector.to(device)
        best = torch.full((len(self.abstract_ids),), -10.0, device=device)
        best = best.scatter_reduce(0, self.chunk_to_abstract, scores, reduce="amax", include_self=True)
        top = best.topk(k).indices.tolist()
        return [self.abstract_ids[i] for i in top]

    def top_chunks(self, query_vector, k):
        scores = self.embeddings @ query_vector.to(device)
        top = scores.topk(k).indices.tolist()
        return top


Writing part_d_rag/rag.py


In [32]:
%%writefile part_d_rag/metrics.py
def hit_at_k(ranked, relevant, k):
    found = set(ranked[:k]) & set(relevant)
    if len(found) > 0:
        return 1.0
    return 0.0


def recall_at_k(ranked, relevant, k):
    relevant = set(relevant)
    found = set(ranked[:k]) & relevant
    return len(found) / min(k, len(relevant))


def mrr_at_10(ranked, relevant):
    relevant = set(relevant)
    for position, abstract_id in enumerate(ranked[:10]):
        if abstract_id in relevant:
            return 1.0 / (position + 1)
    return 0.0


Writing part_d_rag/metrics.py


In [33]:
%%writefile part_d_rag/generate.py
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from rag import GENERATOR_NAME, device

NOTICE = "For educational use only. Not medical advice."


def load_generator():
    tokenizer = AutoTokenizer.from_pretrained(GENERATOR_NAME)
    model = AutoModelForCausalLM.from_pretrained(GENERATOR_NAME)
    if device == "cuda":
        model = model.half()
    model = model.to(device)
    model.eval()
    return tokenizer, model


def build_messages(question, passages):
    numbered = ""
    for i, passage in enumerate(passages):
        numbered = numbered + "[" + str(i + 1) + "] " + passage + "\n\n"

    system = ("You answer biomedical research questions. Use only the numbered passages given by the user. "
              "Cite the passage numbers in square brackets, for example [1]. "
              "If the passages do not contain the answer, say that you do not know.")
    user = "Passages:\n\n" + numbered + "Question: " + question
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def generate_answer(question, passages, tokenizer, model):
    messages = build_messages(question, passages)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


Writing part_d_rag/generate.py


In [34]:
%%writefile part_d_rag/build_index.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_d_rag")

import json
import time
import torch
from common import *
from bioasq import load_corpus, load_questions
from rag import split_into_chunks, load_retriever, embed_texts

make_folders()

corpus = load_corpus()
questions = load_questions("test", corpus)
print("abstracts:", len(corpus), "| test questions:", len(questions))

chunk_abstract_ids = []
chunk_texts = []
for abstract_id in sorted(corpus.keys()):
    for piece in split_into_chunks(corpus[abstract_id]):
        chunk_abstract_ids.append(abstract_id)
        chunk_texts.append(piece)
print("chunks:", len(chunk_texts))

longest = 0
for piece in chunk_texts:
    longest = max(longest, len(piece.split()))
print("longest chunk in words:", longest)

tokenizer, model = load_retriever()
start_time = time.time()
embeddings = embed_texts(chunk_texts, tokenizer, model)
seconds = time.time() - start_time
print("embedding matrix:", tuple(embeddings.shape), "| time in minutes:", round(seconds / 60, 1))

torch.save(embeddings, "cache/embeddings.pt")
file = open("cache/chunks.json", "w", encoding="utf-8")
json.dump({"abstract_ids": chunk_abstract_ids, "texts": chunk_texts}, file)
file.close()

total_relevant = 0
for q in questions:
    total_relevant = total_relevant + len(set(q["relevant_ids"]))

save_json({"abstracts": len(corpus), "chunks": len(chunk_texts), "test_questions": len(questions),
           "mean_relevant": total_relevant / len(questions), "embedding_seconds": seconds,
           "gpu": gpu_name()}, "results/rag_index_info.json")
print("saved cache/embeddings.pt, cache/chunks.json and results/rag_index_info.json")


Writing part_d_rag/build_index.py


In [35]:
!python part_d_rag/build_index.py

abstracts: 40181 | test questions: 707
chunks: 79332
longest chunk in words: 150
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 1.01MB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 14.8MB/s]
tokenizer.json: 100%|████████████████████████| 466k/466k [00:00<00:00, 39.3MB/s]
special_tokens_map.json: 100%|██████████████████| 112/112 [00:00<00:00, 360kB/s]

model.safetensors: downloading bytes: ███████████████▉      | 66.0MB, 5.17MB/s  
model.safetensors: downloading bytes: ██████████████████████| 85.0MB, 8.15MB/s  
model.safetensors: reconstructing file: 100%|██████| 90.9MB / 90.9MB, 8.89MB/s  
Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 2769.94it/s]
embedding matrix: (79332, 384) | time in minutes: 3.4
saved cache/embeddings.pt, cache/chunks.json and results/rag_index_info.json


In [36]:
%%writefile part_d_rag/eval_retrieval.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_d_rag")

import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import *
from bioasq import load_corpus, load_questions
from rag import load_retriever, embed_texts, Index
from metrics import hit_at_k, recall_at_k, mrr_at_10

make_folders()

corpus = load_corpus()
questions = load_questions("test", corpus)
print("test questions:", len(questions))

tokenizer, model = load_retriever()
index = Index()

question_texts = []
for q in questions:
    question_texts.append(q["question"])
question_vectors = embed_texts(question_texts, tokenizer, model)

ranked_lists = []
for i in range(len(questions)):
    ranked_lists.append(index.top_abstracts(question_vectors[i], 20))
print("searched all questions")

table = {}
for k in [1, 5, 10]:
    hits = []
    recalls = []
    for i in range(len(questions)):
        hits.append(hit_at_k(ranked_lists[i], questions[i]["relevant_ids"], k))
        recalls.append(recall_at_k(ranked_lists[i], questions[i]["relevant_ids"], k))
    table["hit@" + str(k)] = mean(hits)
    table["recall@" + str(k)] = mean(recalls)

mrr_values = []
for i in range(len(questions)):
    mrr_values.append(mrr_at_10(ranked_lists[i], questions[i]["relevant_ids"]))
table["mrr@10"] = mean(mrr_values)

recall_curve = []
for k in range(1, 21):
    values = []
    for i in range(len(questions)):
        values.append(recall_at_k(ranked_lists[i], questions[i]["relevant_ids"], k))
    recall_curve.append(mean(values))

for name in table:
    print(name, round(table[name], 4))

failures = []
for i in range(len(questions)):
    if hit_at_k(ranked_lists[i], questions[i]["relevant_ids"], 10) == 0.0:
        relevant_id = questions[i]["relevant_ids"][0]
        top_id = ranked_lists[i][0]
        failures.append({"index": i, "question": questions[i]["question"], "reference_answer": questions[i]["answer"],
                         "relevant_id": relevant_id, "relevant_text": corpus[relevant_id][:400],
                         "top1_id": top_id, "top1_text": corpus[top_id][:400]})
print("questions with no relevant abstract in the top 10:", len(failures))

save_json({"table": table, "recall_curve": recall_curve, "num_questions": len(questions),
           "num_failures": len(failures)}, "results/retrieval.json")
save_json(failures[:15], "results/retrieval_failures.json")

plt.figure(figsize=(6, 4))
plt.plot(range(1, 21), recall_curve, marker="o")
plt.xlabel("k")
plt.ylabel("Recall@k")
plt.title("Retrieval Recall@k")
plt.grid(True)
plt.tight_layout()
plt.savefig("results/fig_rag_recall.png", dpi=150)
print("saved results/fig_rag_recall.png")


Writing part_d_rag/eval_retrieval.py


In [37]:
!python part_d_rag/eval_retrieval.py

test questions: 707
Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 2682.98it/s]
searched all questions
hit@1 0.6223
recall@1 0.6223
hit@5 0.8161
recall@5 0.5863
hit@10 0.8614
recall@10 0.6024
mrr@10 0.7061
questions with no relevant abstract in the top 10: 98
saved results/fig_rag_recall.png


In [38]:
%%writefile part_d_rag/eval_generation.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_d_rag")

import csv
import json
import random
import re
import os
import torch
from rouge_score import rouge_scorer
from common import *
from bioasq import load_corpus, load_questions
from rag import load_retriever, embed_texts, Index
from generate import load_generator, generate_answer

make_folders()

NUM_QUESTIONS = 100
PASSAGES = 5

corpus = load_corpus()
questions = load_questions("test", corpus)[:NUM_QUESTIONS]

retriever_tokenizer, retriever_model = load_retriever()
index = Index()
generator_tokenizer, generator_model = load_generator()

question_vectors = embed_texts([q["question"] for q in questions], retriever_tokenizer, retriever_model)
scorer = rouge_scorer.RougeScorer(["rougeL"])

dont_know_phrases = ["do not know", "don't know", "does not contain", "do not contain",
                     "not enough information", "cannot answer", "can't answer", "no information"]


def find_cited_numbers(answer):
    numbers = []
    for group in re.findall(r"\[([\d,\s]+)\]", answer):
        for part in group.split(","):
            part = part.strip()
            if part.isdigit():
                numbers.append(int(part))
    return numbers


records = []
rouge_values = []
cited_total = 0
cited_correct = 0
dont_know_count = 0

for i in range(len(questions)):
    q = questions[i]
    chunk_ids = index.top_chunks(question_vectors[i], PASSAGES)
    passage_texts = []
    passages = []
    for number, chunk_id in enumerate(chunk_ids):
        passage_texts.append(index.chunk_texts[chunk_id])
        passages.append({"number": number + 1, "abstract_id": index.chunk_abstract_ids[chunk_id],
                         "text": index.chunk_texts[chunk_id]})

    answer = generate_answer(q["question"], passage_texts, generator_tokenizer, generator_model)

    rouge_values.append(scorer.score(q["answer"], answer)["rougeL"].fmeasure)

    cited = find_cited_numbers(answer)
    for number in cited:
        if 1 <= number <= PASSAGES:
            cited_total = cited_total + 1
            if passages[number - 1]["abstract_id"] in q["relevant_ids"]:
                cited_correct = cited_correct + 1

    said_dont_know = False
    for phrase in dont_know_phrases:
        if phrase in answer.lower():
            said_dont_know = True
    if said_dont_know:
        dont_know_count = dont_know_count + 1

    top10 = index.top_abstracts(question_vectors[i], 10)
    retrieved = False
    for abstract_id in top10:
        if abstract_id in q["relevant_ids"]:
            retrieved = True

    records.append({"index": i, "qid": q["qid"], "question": q["question"], "reference_answer": q["answer"],
                    "generated_answer": answer, "cited": cited, "passages": passages,
                    "relevant_ids": q["relevant_ids"], "relevant_in_top10": retrieved,
                    "said_dont_know": said_dont_know})
    print("answered", i + 1, "of", len(questions))

file = open("results/rag_answers.jsonl", "w", encoding="utf-8")
for record in records:
    file.write(json.dumps(record) + "\n")
file.close()

if cited_total > 0:
    citation_precision = cited_correct / cited_total
else:
    citation_precision = 0.0

metrics = {"rouge_l": mean(rouge_values), "citation_precision": citation_precision,
           "cited_total": cited_total, "dont_know_rate": dont_know_count / len(records),
           "questions": len(records)}
save_json(metrics, "results/generation.json")
print(metrics)

chosen = random.Random(0).sample(range(len(records)), 20)
chosen.sort()

lines = ["# RAG answers to be marked by both members", "",
         "For each item write supported (yes/no) and correct (yes/no) in results/human_eval.csv.", ""]
for item in chosen:
    r = records[item]
    lines.append("## Item " + str(item))
    lines.append("")
    lines.append("Question: " + r["question"])
    lines.append("")
    lines.append("Answer: " + r["generated_answer"])
    lines.append("")
    lines.append("Reference answer: " + r["reference_answer"])
    lines.append("")
    for p in r["passages"]:
        lines.append("[" + str(p["number"]) + "] (abstract " + str(p["abstract_id"]) + ") " + p["text"])
        lines.append("")
file = open("results/human_eval_rag_sheet.md", "w", encoding="utf-8")
file.write("\n".join(lines))
file.close()

columns = ["kind", "item", "text", "member1_answer", "member2_answer",
           "member1_supported", "member2_supported", "member1_correct", "member2_correct"]
old_rows = []
if os.path.exists("results/human_eval.csv"):
    file = open("results/human_eval.csv", encoding="utf-8")
    for row in csv.DictReader(file):
        if row["kind"] != "rag":
            old_rows.append(row)
    file.close()

file = open("results/human_eval.csv", "w", newline="", encoding="utf-8")
writer = csv.DictWriter(file, fieldnames=columns)
writer.writeheader()
for row in old_rows:
    writer.writerow(row)
for item in chosen:
    row = {}
    for c in columns:
        row[c] = ""
    row["kind"] = "rag"
    row["item"] = item
    row["text"] = records[item]["question"]
    writer.writerow(row)
file.close()
print("saved results/rag_answers.jsonl, results/human_eval_rag_sheet.md and rag rows in results/human_eval.csv")

failed = []
unsure = []
for r in records:
    if not r["relevant_in_top10"]:
        failed.append(r)
    if r["said_dont_know"]:
        unsure.append(r)

wanted = []
if len(failed) > 0:
    wanted.append(failed[0])
if len(unsure) > 0 and unsure[0] not in wanted:
    wanted.append(unsure[0])
for r in records:
    if len(wanted) < 5 and r not in wanted:
        wanted.append(r)

lines = ["# Qualitative samples", ""]
for r in wanted:
    lines.append("## Question " + str(r["index"]))
    lines.append("")
    lines.append("Question: " + r["question"])
    lines.append("")
    lines.append("Answer: " + r["generated_answer"])
    lines.append("")
    lines.append("Reference answer: " + r["reference_answer"])
    lines.append("")
    lines.append("Relevant abstract found in top 10: " + str(r["relevant_in_top10"]) +
                 " | model said it did not know: " + str(r["said_dont_know"]))
    lines.append("")
    for p in r["passages"]:
        lines.append("[" + str(p["number"]) + "] (abstract " + str(p["abstract_id"]) + ") " + p["text"][:300])
        lines.append("")
file = open("results/samples.md", "w", encoding="utf-8")
file.write("\n".join(lines))
file.close()
print("saved results/samples.md")


Writing part_d_rag/eval_generation.py


In [39]:
!python part_d_rag/eval_generation.py

config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 2.23MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 16.6MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 51.8MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 121MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 161MB/s]

model.safetensors: downloading bytes:                       |  0.00B            
model.safetensors: downloading bytes: ▍                     | 60.0MB, 4.00MB/s  
model.safetensors: downloading bytes: █                     |  152MB, 11.2MB/s  
model.safetensors: downloading bytes: █▎                    |  185MB, 14.3MB/s  
model.safetensors: reconstructing file:   7%|▍     |  209MB / 3.09GB, 16.6MB/s  
model.safetensors: downloading bytes: █▋                    |  239MB, 18.5MB/s  
model.safetensors: downloading bytes: █▊                    |  262MB, 21.7MB/s  
model.safetensors: download

## 8. Correctness checks, human evaluation and tables

In [40]:
%%writefile tools/run_checks.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_a_mae")
sys.path.append("part_b_vae")
sys.path.append("part_c_diffusion")
sys.path.append("part_d_rag")

import os
import json
import torch
from common import *
from eurosat import patchify
from mae_model import MAE
from vae_model import kl_term
from schedule import T, alpha_bars, add_noise
from metrics import hit_at_k, recall_at_k, mrr_at_10

torch.manual_seed(0)
make_folders()

lines = []


def report(text):
    print(text)
    lines.append(text)


report("----- MAE -----")
model = MAE()
images = torch.rand(6, 3, 64, 64)
patches = patchify(images)
loss, pred, mask = model(patches)
masked_per_image = mask.sum(dim=1).tolist()
report("masked patches in each image: " + str(masked_per_image))
report("exactly 48 of 64 masked in every image: " + str(all(m == 48 for m in masked_per_image)))

changed = pred.detach().clone()
visible_position = (mask[0] == 0).nonzero()[0].item()
changed[0, visible_position] = changed[0, visible_position] + 100.0
loss_visible_changed = model.masked_loss(changed, patches, mask)
report("loss unchanged when a visible patch prediction is changed: " +
       str(torch.allclose(loss, loss_visible_changed)))

changed = pred.detach().clone()
masked_position = (mask[0] == 1).nonzero()[0].item()
changed[0, masked_position] = changed[0, masked_position] + 10.0
loss_masked_changed = model.masked_loss(changed, patches, mask)
report("loss changes when a masked patch prediction is changed: " +
       str(not torch.allclose(loss, loss_masked_changed)))

report("")
report("----- VAE -----")
zero = torch.zeros(5, 16)
report("KL when mu = 0 and logvar = 0: " + str(kl_term(zero, zero).item()))
smallest = 1000.0
for i in range(200):
    mu = torch.randn(64, 16) * 3
    logvar = torch.randn(64, 16) * 2
    each = -0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(dim=1)
    smallest = min(smallest, each.min().item())
report("smallest KL value over 12800 random cases: " + str(round(smallest, 6)) + " (never negative: " + str(smallest >= -1e-4) + ")")

report("")
report("----- DDPM -----")
report("alpha_bar at the last step: " + str(alpha_bars[T - 1].item()) + " (about 4e-5)")
x0 = torch.rand(1, 1, 28, 28) * 2 - 1
noise = torch.randn_like(x0)
x_last = add_noise(x0, torch.tensor([T - 1]), noise)
correlation_last = torch.corrcoef(torch.stack([x0.flatten(), x_last.flatten()]))[0, 1].item()
report("correlation between x0 and the noised image at t = 999: " + str(round(correlation_last, 4)) + " (close to 0, looks like noise)")
x_first = add_noise(x0, torch.tensor([0]), noise)
correlation_first = torch.corrcoef(torch.stack([x0.flatten(), x_first.flatten()]))[0, 1].item()
report("correlation between x0 and the noised image at t = 0: " + str(round(correlation_first, 4)) + " (close to 1)")

report("")
report("----- RAG -----")
all_good = True
for size in [1, 2, 5, 15]:
    relevant = list(range(100, 100 + size))
    ranked = relevant + list(range(500, 520))
    for k in [1, 5, 10]:
        if hit_at_k(ranked, relevant, k) != 1.0 or recall_at_k(ranked, relevant, k) != 1.0:
            all_good = False
    if mrr_at_10(ranked, relevant) != 1.0:
        all_good = False
report("perfect retriever gives Hit@k = Recall@k = MRR@10 = 1: " + str(all_good))

if os.path.exists("cache/chunks.json") and os.path.exists("cache/embeddings.pt"):
    file = open("cache/chunks.json", encoding="utf-8")
    data = json.load(file)
    file.close()
    longest = 0
    for text in data["texts"]:
        longest = max(longest, len(text.split()))
    report("longest chunk in words: " + str(longest) + " (at most 150: " + str(longest <= 150) + ")")
    embeddings = torch.load("cache/embeddings.pt")
    norms = embeddings.norm(dim=1)
    report("embedding norms: min " + str(round(norms.min().item(), 5)) + ", max " + str(round(norms.max().item(), 5)))
else:
    report("chunk and embedding checks skipped: run part_d_rag/build_index.py first")

file = open("results/correctness_checks.txt", "w")
file.write("\n".join(lines))
file.close()
print("saved results/correctness_checks.txt")


Writing tools/run_checks.py


In [41]:
!python tools/run_checks.py

----- MAE -----
masked patches in each image: [48.0, 48.0, 48.0, 48.0, 48.0, 48.0]
exactly 48 of 64 masked in every image: True
loss unchanged when a visible patch prediction is changed: True
loss changes when a masked patch prediction is changed: True

----- VAE -----
KL when mu = 0 and logvar = 0: 0.0
smallest KL value over 12800 random cases: 24.894226 (never negative: True)

----- DDPM -----
alpha_bar at the last step: 4.035830352222547e-05 (about 4e-5)
correlation between x0 and the noised image at t = 999: 0.0829 (close to 0, looks like noise)
correlation between x0 and the noised image at t = 0: 0.9999 (close to 1)

----- RAG -----
perfect retriever gives Hit@k = Recall@k = MRR@10 = 1: True
longest chunk in words: 150 (at most 150: True)
embedding norms: min 1.0, max 1.0
saved results/correctness_checks.txt


In [42]:
%%writefile tools/human_eval.py
import sys
sys.path.append(".")

import csv
from common import *

key = load_json("results/blind_test_key.json")

file = open("results/human_eval.csv", encoding="utf-8")
rows = list(csv.DictReader(file))
file.close()


def is_yes(text):
    return text.strip().lower() in ["y", "yes", "1", "true"]


correct1 = 0
correct2 = 0
agreed = 0
diffusion_rows = 0
empty = 0

for row in rows:
    if row["kind"] != "diffusion":
        continue
    diffusion_rows = diffusion_rows + 1
    truth = key[str(row["item"])].strip().lower()
    answer1 = row["member1_answer"].strip().lower()
    answer2 = row["member2_answer"].strip().lower()
    if answer1 == "" or answer2 == "":
        empty = empty + 1
    if answer1 == truth:
        correct1 = correct1 + 1
    if answer2 == truth:
        correct2 = correct2 + 1
    if answer1 == answer2:
        agreed = agreed + 1

supported1 = 0
supported2 = 0
right1 = 0
right2 = 0
rag_rows = 0
for row in rows:
    if row["kind"] != "rag":
        continue
    rag_rows = rag_rows + 1
    if is_yes(row["member1_supported"]):
        supported1 = supported1 + 1
    if is_yes(row["member2_supported"]):
        supported2 = supported2 + 1
    if is_yes(row["member1_correct"]):
        right1 = right1 + 1
    if is_yes(row["member2_correct"]):
        right2 = right2 + 1

print("diffusion blind test (" + str(diffusion_rows) + " images, " + str(empty) + " with a missing answer)")
print("member 1 correct:", correct1, "| member 2 correct:", correct2)
print("images where both members agreed:", agreed)
print("")
print("RAG human check (" + str(rag_rows) + " answers)")
print("supported, member 1 / member 2:", supported1, "/", supported2)
print("correct, member 1 / member 2:", right1, "/", right2)

save_json({"diffusion_images": diffusion_rows, "member1_correct": correct1, "member2_correct": correct2,
           "both_agreed": agreed, "rag_items": rag_rows,
           "supported_member1": supported1, "supported_member2": supported2,
           "correct_member1": right1, "correct_member2": right2}, "results/human_eval_scores.json")
print("saved results/human_eval_scores.json")


Writing tools/human_eval.py


In [53]:
%%writefile tools/make_tables.py
import sys
sys.path.append(".")

import os
from common import *


def get(path):
    if os.path.exists(path):
        return load_json(path)
    return None


def r(value, digits=4):
    return str(round(value, digits))


mae = get("results/mae_pretrain.json")
clf = get("results/mae_classifier.json")
vae = get("results/vae_train.json")
diff = get("results/diffusion_train.json")
index_info = get("results/rag_index_info.json")
retrieval = get("results/retrieval.json")
generation = get("results/generation.json")
human = get("results/human_eval_scores.json")

out = []


def table(title, header, rows):
    out.append("## " + title)
    out.append("| " + " | ".join(header) + " |")
    out.append("|" + "---|" * len(header))
    for row in rows:
        out.append("| " + " | ".join(row) + " |")
    out.append("")


rows = []
if mae and clf:
    rows.append(["EuroSAT train / test images; labelled images used",
                 str(mae["train_images"]) + " / " + str(clf["test_images"]) + "; " + str(clf["labelled_images"])])
if vae:
    rows.append(["UHaT classes; train / test images",
                 str(vae["classes"]) + "; " + str(vae["train_images"]) + " / " + str(vae["test_images"])])
if index_info:
    rows.append(["BioASQ abstracts; chunks; test questions",
                 str(index_info["abstracts"]) + "; " + str(index_info["chunks"]) + "; " + str(index_info["test_questions"])])
    rows.append(["Mean relevant abstracts per question", r(index_info["mean_relevant"], 2)])
table("Table 1 - Data", ["Item", "Value"], rows)

rows = []
if mae:
    rows.append(["MAE", str(mae["params"]), r(mae["epoch_losses"][-1]), r(mae["seconds"] / 60, 1) + " min / " + mae["gpu"]])
if vae:
    rows.append(["VAE", str(vae["params"]), r(vae["train_total"][-1], 2), r(vae["seconds"] / 60, 1) + " min / " + vae["gpu"]])
if diff:
    rows.append(["DDPM U-Net", str(diff["params"]), r(diff["epoch_losses"][-1]), r(diff["seconds"] / 60, 1) + " min / " + diff["gpu"]])
table("Table 2 - Training", ["Model", "Parameters", "Final loss", "Time / GPU"], rows)

rows = []
if clf:
    rows.append(["MAE classifier test accuracy", r(clf["test_accuracy"])])
if vae:
    rows.append(["VAE test loss = recon + KL",
                 r(vae["test_total"][-1], 2) + " = " + r(vae["test_recon"][-1], 2) + " + " + r(vae["test_kl"][-1], 2)])
if human:
    rows.append(["Diffusion blind test correct (member 1 / member 2, of 40)",
                 str(human["member1_correct"]) + " / " + str(human["member2_correct"])])
    rows.append(["Diffusion images both agreed on (of 40)", str(human["both_agreed"])])
table("Table 3 - Image results", ["Item", "Value"], rows)

if retrieval:
    t = retrieval["table"]
    names = ["hit@1", "hit@5", "hit@10", "recall@1", "recall@5", "recall@10", "mrr@10"]
    table("Table 4 - Retrieval", names, [[r(t[name]) for name in names]])

rows = []
if generation:
    rows.append(["ROUGE-L", r(generation["rouge_l"])])
    rows.append(["Citation precision", r(generation["citation_precision"])])
    rows.append(["I don't know rate", r(generation["dont_know_rate"])])
if human:
    rows.append(["Supported (member 1 / member 2, of 20)",
                 str(human["supported_member1"]) + " / " + str(human["supported_member2"])])
    rows.append(["Correct (member 1 / member 2, of 20)",
                 str(human["correct_member1"]) + " / " + str(human["correct_member2"])])
table("Table 5 - Generation", ["Item", "Value"], rows)

file = open("results/tables.md", "w", encoding="utf-8")
file.write("\n".join(out))
file.close()
for line in out:
    if not line.startswith("|---"):
        print(line)

Overwriting tools/make_tables.py


Both members now fill `results/human_eval.csv` (download it, fill it, upload it back to `results/`). The RAG answers to mark are in `results/human_eval_rag_sheet.md`. Then run the next two cells.

In [54]:
!python tools/human_eval.py

diffusion blind test (40 images, 40 with a missing answer)
member 1 correct: 0 | member 2 correct: 0
images where both members agreed: 40

RAG human check (20 answers)
supported, member 1 / member 2: 0 / 0
correct, member 1 / member 2: 0 / 0
saved results/human_eval_scores.json


In [55]:
!python tools/make_tables.py

## Table 1 - Data
| Item | Value |
| EuroSAT train / test images; labelled images used | 16200 / 5400; 160 |
| UHaT classes; train / test images | 40; 28328 / 4747 |
| BioASQ abstracts; chunks; test questions | 40181; 79332; 707 |
| Mean relevant abstracts per question | 8.66 |

## Table 2 - Training
| Model | Parameters | Final loss | Time / GPU |
| MAE | 3173248 | 0.0064 | 7.5 min / Tesla T4 |
| VAE | 220385 | 130.43 | 0.5 min / Tesla T4 |
| DDPM U-Net | 4028353 | 0.0229 | 34.1 min / Tesla T4 |

## Table 3 - Image results
| Item | Value |
| MAE classifier test accuracy | 0.5281 |
| VAE test loss = recon + KL | 130.62 = 104.54 + 26.09 |
| Diffusion blind test correct (member 1 / member 2, of 40) | 0 / 0 |
| Diffusion images both agreed on (of 40) | 40 |

## Table 4 - Retrieval
| hit@1 | hit@5 | hit@10 | recall@1 | recall@5 | recall@10 | mrr@10 |
| 0.6223 | 0.8161 | 0.8614 | 0.6223 | 0.5863 | 0.6024 | 0.7061 |

## Table 5 - Generation
| Item | Value |
| ROUGE-L | 0.2538 |
| Citation pr

## 9. The app

The app is written here so that it ends up in your repository. It is run on your own computer with `python app/app.py` after the weights are downloaded (see the README).

In [46]:
%%writefile app/app.py
import sys
sys.path.append(".")
sys.path.append("starter")
sys.path.append("part_a_mae")
sys.path.append("part_b_vae")
sys.path.append("part_c_diffusion")
sys.path.append("part_d_rag")

import numpy as np
import torch
import torch.nn as nn
import gradio as gr
from common import *
from eurosat import load_eurosat_split, patchify, unpatchify, CLASS_NAMES
from mae_model import MAE
from vae_model import VAE
from unet import make_unet
from sample import sample_images, to_01
from rag import load_retriever, embed_texts, Index
from generate import load_generator, generate_answer, NOTICE

store = {}


def get(name, loader):
    if name not in store:
        store[name] = loader()
    return store[name]


def to_picture(images, ncols, scale):
    images = images.detach().cpu().clamp(0, 1)
    n, c, h, w = images.shape
    rows = (n + ncols - 1) // ncols
    canvas = np.zeros((rows * h, ncols * w, 3), dtype=np.float32)
    for i in range(n):
        r = i // ncols
        col = i % ncols
        tile = images[i].permute(1, 2, 0).numpy()
        if c == 1:
            tile = np.repeat(tile, 3, axis=2)
        canvas[r * h:(r + 1) * h, col * w:(col + 1) * w] = tile
    canvas = np.repeat(np.repeat(canvas, scale, axis=0), scale, axis=1)
    return (canvas * 255).astype(np.uint8)


def load_mae_parts():
    model = MAE().to(device)
    model.load_state_dict(torch.load("weights/mae.pt", map_location=device))
    model.eval()
    classifier = nn.Linear(192, 10).to(device)
    classifier.load_state_dict(torch.load("weights/mae_classifier.pt", map_location=device))
    images, labels = load_eurosat_split("test")
    return model, classifier, images, labels


def run_mae(tile_number):
    model, classifier, images, labels = get("mae", load_mae_parts)
    tile_number = int(tile_number)
    image = images[tile_number:tile_number + 1]
    patches = patchify(image)
    with torch.no_grad():
        loss, pred, mask = model(patches.to(device))
        features = model.features(patches.to(device))
        predicted_class = classifier(features).argmax(dim=1).item()
    pred = pred.cpu()
    mask3 = mask.cpu().unsqueeze(-1)
    masked_input = unpatchify(patches * (1 - mask3) + 0.5 * mask3)
    reconstruction = unpatchify(patches * (1 - mask3) + pred * mask3)
    text = "Predicted class: " + CLASS_NAMES[predicted_class] + "  |  True class: " + CLASS_NAMES[labels[tile_number].item()]
    return to_picture(masked_input, 1, 4), to_picture(reconstruction, 1, 4), to_picture(image, 1, 4), text


def load_vae_parts():
    model = VAE().to(device)
    model.load_state_dict(torch.load("weights/vae.pt", map_location=device))
    model.eval()
    x_train, y_train, x_test, y_test, names = load_uhat()
    return model, x_test, y_test, names


def new_characters():
    model, x_test, y_test, names = get("vae", load_vae_parts)
    z = torch.randn(64, 16).to(device)
    with torch.no_grad():
        samples = torch.sigmoid(model.decode(z))
    return to_picture(samples, 8, 3)


def interpolate(first_name, second_name, amount):
    model, x_test, y_test, names = get("vae", load_vae_parts)
    first = names.index(first_name)
    second = names.index(second_name)
    a = x_test[(y_test == first).nonzero()[0].item()].unsqueeze(0).to(device)
    b = x_test[(y_test == second).nonzero()[0].item()].unsqueeze(0).to(device)
    with torch.no_grad():
        mu_a, _ = model.encode(a)
        mu_b, _ = model.encode(b)
        z = (1 - amount) * mu_a + amount * mu_b
        result = torch.sigmoid(model.decode(z))
    return to_picture(torch.cat([a.cpu(), result.cpu(), b.cpu()], dim=0), 3, 8)


def load_unet():
    unet = make_unet().to(device)
    unet.load_state_dict(torch.load("weights/unet.pt", map_location=device))
    unet.eval()
    return unet


def write_character(name):
    unet = get("unet", load_unet)
    names = load_json("results/diffusion_train.json")["class_names"]
    label = torch.full((8,), names.index(name), device=device, dtype=torch.long)
    samples, saved = sample_images(unet, label)
    return to_picture(to_01(samples), 8, 4)


def load_rag_parts():
    retriever_tokenizer, retriever_model = load_retriever()
    index = Index()
    generator_tokenizer, generator_model = load_generator()
    return retriever_tokenizer, retriever_model, index, generator_tokenizer, generator_model


def answer_question(question):
    retriever_tokenizer, retriever_model, index, generator_tokenizer, generator_model = get("rag", load_rag_parts)
    vector = embed_texts([question], retriever_tokenizer, retriever_model)[0]
    chunk_ids = index.top_chunks(vector, 5)
    passages = []
    for chunk_id in chunk_ids:
        passages.append(index.chunk_texts[chunk_id])
    answer = generate_answer(question, passages, generator_tokenizer, generator_model)

    shown = ""
    for number, chunk_id in enumerate(chunk_ids):
        shown = shown + "**[" + str(number + 1) + "]** abstract id " + str(index.chunk_abstract_ids[chunk_id]) + "\n\n"
        shown = shown + passages[number] + "\n\n"
    return answer + "\n\n*" + NOTICE + "*", shown


vae_names = load_json("results/vae_train.json")["class_names"] if os.path.exists("results/vae_train.json") else ["alif"]
diffusion_names = vae_names

with gr.Blocks(title="Four generative models") as demo:
    gr.Markdown("# Four generative models in one app")

    with gr.Tab("MAE (satellite images)"):
        gr.Markdown("Pick a test tile. The model sees only 25% of the patches and rebuilds the rest.")
        tile = gr.Slider(0, 5399, value=0, step=1, label="Test tile number")
        mae_button = gr.Button("Run")
        with gr.Row():
            masked_out = gr.Image(label="Masked input")
            recon_out = gr.Image(label="Reconstruction")
            original_out = gr.Image(label="Original")
        mae_text = gr.Textbox(label="Land cover class")
        mae_button.click(run_mae, inputs=tile, outputs=[masked_out, recon_out, original_out, mae_text])

    with gr.Tab("VAE (Urdu characters)"):
        vae_button = gr.Button("New random characters")
        vae_out = gr.Image(label="64 new characters")
        vae_button.click(new_characters, outputs=vae_out)
        with gr.Row():
            first_choice = gr.Dropdown(vae_names, value=vae_names[0], label="First character")
            second_choice = gr.Dropdown(vae_names, value=vae_names[1], label="Second character")
        amount = gr.Slider(0, 1, value=0.5, step=0.1, label="Morph amount")
        morph_out = gr.Image(label="First, morph, second")
        amount.change(interpolate, inputs=[first_choice, second_choice, amount], outputs=morph_out)

    with gr.Tab("Diffusion (write a character)"):
        character = gr.Dropdown(diffusion_names, value=diffusion_names[0], label="Character to write")
        diffusion_button = gr.Button("Generate 8 samples")
        diffusion_out = gr.Image(label="8 new samples")
        diffusion_button.click(write_character, inputs=character, outputs=diffusion_out)

    with gr.Tab("RAG (biomedical questions)"):
        gr.Markdown("**" + NOTICE + "**")
        question_box = gr.Textbox(label="Your question")
        rag_button = gr.Button("Answer")
        answer_out = gr.Markdown()
        passages_out = gr.Markdown()
        rag_button.click(answer_question, inputs=question_box, outputs=[answer_out, passages_out])

if __name__ == "__main__":
    demo.launch()


Writing app/app.py


## 10. Save everything

This cell makes the two zip files. Everything in /kaggle/working is also kept in the Output tab of the saved version, so after Save & Run All you can download results.zip and weights.zip from there. If you run the notebook by hand, click the two links below.

In [47]:
!zip -q -r results.zip results
!zip -q -r weights.zip weights
!ls -lh *.zip
from IPython.display import FileLink, display
display(FileLink('results.zip'))
display(FileLink('weights.zip'))

-rw-r--r-- 1 root root  32M Nov 25 06:59 data.zip
-rw-r--r-- 1 root root 2.2M Nov 25 07:57 results.zip
-rw-r--r-- 1 root root  27M Nov 25 07:57 weights.zip


/kaggle/working/results.zip

/kaggle/working/weights.zip